# Universal Transformer: повторное ядро

**Universal Transformers** · Dehghani et al. · 2018

[Оригинальная статья](https://arxiv.org/abs/1807.03819)

**Цель урока:** Проверить sharing, depth signals и разницу полного и обрезанного backward.

## Перед началом

Attention, residual blocks и вводный урок looped transformers.

Около 40–60 минут с заданиями. CPU, PyTorch и Matplotlib; без скачивания весов и датасетов. Время — ориентир для чтения и работы, а не benchmark выполнения.

## Задача и идея

Universal Transformer повторяет общее преобразование по глубине, сохраняя параллельную обработку позиций. Модель может получать сигнал позиции и шага вычисления. В статье также рассматривается адаптивная остановка для разных позиций.

Повторное использование одного объекта nn.Module — sharing; создание нескольких одинаково инициализированных объектов — нет. Глубина вычисления, число параметров и длина backward — разные величины. Ниже исследуем fixed-depth вариант: adaptive computation time намеренно не реализуется.

## Граница воспроизведения

Fixed-depth shared Transformer; нет adaptive halting, полного оригинального position/depth encoding и задач статьи.

Код написан для этого урока по математическим идеям статьи. Это не скопированный авторский репозиторий. Успех assertions подтверждает указанные свойства реализации, но не результаты статьи.

## Формула → тензоры

`h_(s+1) = Core_theta(h_s + depth_signal(s))`. Theta остаётся одной и той же на каждом шаге.

Full BPTT суммирует пути градиента через все применения. `detach()` отсекает предыдущую историю; это другой estimator градиента. Checkpointing вместо detach может экономить activation memory через пересчёт, сохраняя математический градиент при согласованном RNG.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Шаг 1

Одно causal ядро получает детерминированный depth signal. Он не добавляет обучаемых параметров на каждый проход.

In [ ]:
class SharedCore(nn.Module):
    def __init__(self, width=12):
        super().__init__()
        self.core = nn.TransformerEncoderLayer(width, 3, 24, dropout=0, batch_first=True)
        self.width = width
    def forward(self, x, depth, truncate=False):
        h = x
        causal = torch.triu(torch.ones(x.shape[1], x.shape[1], dtype=torch.bool), 1)
        for index in range(depth):
            if truncate and index:
                h = h.detach()
            signal = torch.sin(torch.arange(self.width) / self.width + index) * 0.1
            h = self.core(h + signal, src_mask=causal)
        return h
model = SharedCore(); x = torch.randn(2, 5, 12)
count = sum(p.numel() for p in model.parameters())
for depth in [1, 2, 4]:
    print('depth:', depth, 'parameters:', count, 'output:', model(x, depth).shape)
assert len(list(model.children())) == 1

## Шаг 2

Сравниваем градиенты при абсолютно одинаковом forward. Отличается только обратный проход.

In [ ]:
target = torch.randn_like(x)
def gradient(truncate):
    model.zero_grad(set_to_none=True)
    output = model(x, 4, truncate)
    F.mse_loss(output, target).backward()
    return output.detach(), model.core.linear1.weight.grad.clone()
full_output, full_gradient = gradient(False)
truncated_output, truncated_gradient = gradient(True)
assert torch.equal(full_output, truncated_output)
print('gradient difference:', (full_gradient - truncated_gradient).norm().item())
assert not torch.allclose(full_gradient, truncated_gradient)

## Шаг 3

Учим на глубине 2, измеряем на нескольких глубинах. Большая глубина не обязана улучшать результат.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
for step in range(40):
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(model(x, 2), target); loss.backward(); optimizer.step()
with torch.no_grad():
    depths = [1, 2, 3, 4, 6]
    losses = [F.mse_loss(model(x, d), target).item() for d in depths]
print(dict(zip(depths, losses)))
plt.plot(depths, losses, marker='o', color='#d73535'); plt.xlabel('inference depth'); plt.ylabel('training-batch MSE'); plt.show()

## Как читать результат

Loss измеряется на том же маленьком batch: это проверка влияния глубины, не generalization benchmark. Равные outputs при разных градиентах показывают, почему detach нельзя считать безвредной оптимизацией памяти.

## Самостоятельные эксперименты

1. Замените общее ядро на четыре независимых.
2. Уберите depth signal.
3. Сделайте train depth случайной от 1 до 4 и сравните по отдельной выборке.

Сначала запишите прогноз, затем меняйте по одному фактору. Для честного сравнения сбрасывайте ядро и выполняйте все ячейки с начала. Сохраните копию для своих изменений.

## Проверка рассуждений — после эксперимента

Независимые ядра увеличивают число параметров. Без depth signal шаг вычисления известен только через evolving state. Random depth меняет распределение задач обучения; нужны одинаковый compute budget и несколько seed.

## Что записать в лабораторный журнал

- Гипотеза и изменённый параметр.
- Seed, размер данных, число шагов и используемая precision.
- Значения проверок до и после, график и объяснение расхождения с прогнозом.
- Что этот эксперимент **не** позволяет заключить.

Связанный раздел доступен по кнопке «Открыть раздел студии» над уроком.